# Cálculo de VaR con TimeGAN y Simulación de Monte Carlo

Este notebook implementa un flujo completo para:
1. Cargar y preprocesar datos históricos de precios de acciones
2. Entrenar un modelo TimeGAN
3. Generar datos sintéticos futuros
4. Calcular el VaR mediante simulación de Monte Carlo

In [4]:
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
import tensorflow as tf

# Añadir el directorio raíz al path
sys.path.append(os.path.abspath('..'))

# Importar módulos del proyecto
from src.data.loader import load_stock_data, check_data_quality
from src.data.transform import calculate_returns, add_features, normalize_data, inverse_normalize
from src.data.windowing import create_windows, recreate_time_series
from src.models.timegan import timegan_init, timegan_train, timegan_export_generator, generator_gen, generator_save, generator_load
from src.models.var_model import monte_carlo_var, parametric_var, historical_var
from src.utils.evaluation import calculate_kl_divergence, discriminative_score, predictive_score, visualize_tsne
from src.utils.visualization import plot_stock_prices, plot_returns_distribution, plot_acf_comparison, plot_var_histogram, plot_multiple_var_methods

# Configurar semilla para reproducibilidad
np.random.seed(42)
tf.random.set_seed(42)

## 1. Descargar y Preprocesar Datos

Descargamos toda la historia de los activos que incluiremos en el portafolio. Es una estrategia Naive compuesta por los principales indices de EE.UU y Europa. 

In [7]:
## Descargar la data
#yf.Ticker("^GSPC").history(period="max").to_csv('../data/input/sp500.csv')
#yf.Ticker("^STOXX").history(period="max").to_csv('../data/input/stoxx.csv')


In [ ]:
# Cargar datos descargados
file_path = '../data/raw/stock_prices.csv'  # Ajustar ruta según sea necesario
df = load_stock_data(file_path)
df = check_data_quality(df)

# Mostrar información básica
print(f"Período de datos: {df.index[0]} a {df.index[-1]}")
print(f"Número total de observaciones: {len(df)}")
df

In [ ]:
# Calcular rendimientos y añadir características
df = calculate_returns(df, method='log')
df = add_features(df, window_sizes=[5, 10, 20])

# Visualizar las primeras filas después del preprocesamiento
df.head()

In [ ]:
# Normalizar los datos para el entrenamiento del modelo
features_to_normalize = [col for col in df.columns if col != 'Close']
df_norm, norm_params = normalize_data(df, features=features_to_normalize, method='minmax')

# Visualizar datos normalizados
df_norm.head()

In [ ]:
# Crear ventanas temporales para el entrenamiento
window_size = 30  # 30 días de datos históricos
stride = 5       # Avanzar de 5 en 5 días para crear ventanas

# Seleccionar características para el modelo
selected_features = ['Returns', 'Volatility_5', 'Volatility_10', 'MA_5', 'MA_10']
windows = create_windows(df_norm[selected_features], window_size=window_size, stride=stride)

print(f"Forma de las ventanas de entrenamiento: {windows.shape}")

## 2. Entrenar el Modelo TimeGAN

In [ ]:
# Definir parámetros del modelo
time_series_len = window_size
features = len(selected_features)
rnn_units = 64   # Unidades en cada capa GRU
rnn_layers = 3   # Número de capas GRU

# Inicializar el modelo TimeGAN
timegan_tuple = timegan_init(time_series_len, features, rnn_units, rnn_layers)

# Parámetros de entrenamiento
epochs = 100      # Reducido para demostración, usar 1000-2000 para resultados óptimos
batch_size = 32
learning_rate = 0.001

# Entrenar el modelo (esto puede tomar tiempo)
trained_timegan = timegan_train(windows, timegan_tuple, epochs, batch_size, learning_rate)

# Exportar el generador
syn_generator = timegan_export_generator(trained_timegan)

# Guardar el modelo entrenado
os.makedirs('../models', exist_ok=True)
generator_save(syn_generator, '../models/timegan_stock_model.h5')

## 3. Generar Datos Sintéticos y Evaluar

In [ ]:
# Cargar el modelo guardado (opcional, si se quiere saltar el entrenamiento)
# syn_generator = generator_load('../models/timegan_stock_model.h5', time_series_len, features, rnn_units, rnn_layers)

# Generar datos sintéticos
n_simulations = 100  # Número de trayectorias sintéticas
synthetic_windows = generator_gen(syn_generator, generate_cnt=n_simulations)

print(f"Forma de las ventanas sintéticas: {synthetic_windows.shape}")

In [ ]:
# Evaluar la calidad de los datos sintéticos
kl_div = calculate_kl_divergence(windows, synthetic_windows)
disc_score = discriminative_score(windows, synthetic_windows)
pred_score = predictive_score(windows, synthetic_windows)

print(f"Divergencia KL: {kl_div:.4f}")
print(f"Discriminative Score: {disc_score:.4f}")
print(f"Predictive Score (MSE): {pred_score:.4f}")

# Visualizar datos con t-SNE
tsne_fig = visualize_tsne(windows, synthetic_windows)
plt.show()

## 4. Proyectar Trayectorias Futuras y Reconstruir Series de Precios

In [ ]:
# Desnormalizar las ventanas sintéticas
synthetic_df_list = []

for i in range(n_simulations):
    # Crear un DataFrame con las columnas seleccionadas
    synth_df = pd.DataFrame(synthetic_windows[i], columns=selected_features)
    
    # Desnormalizar
    synth_df = inverse_normalize(synth_df, norm_params, method='minmax')
    synthetic_df_list.append(synth_df)

# Reconstruir las series de precios a partir de los rendimientos sintéticos
last_price = df['Close'].iloc[-1]
horizon = 100  # Días futuros a proyectar

future_prices_list = []
future_dates = pd.date_range(start=df.index[-1] + pd.Timedelta(days=1), periods=horizon)

for synth_df in synthetic_df_list:
    # Usar solo los primeros 'horizon' puntos de las ventanas
    returns = synth_df['Returns'].values[:horizon]
    
    # Reconstruir precios a partir de rendimientos logarítmicos
    future_prices = [last_price]
    for ret in returns:
        future_prices.append(future_prices[-1] * np.exp(ret))
    
    future_prices = future_prices[1:]  # Eliminar el precio inicial que ya tenemos
    future_prices_list.append(future_prices)

# Visualizar trayectorias de precios históricos y sintéticos
plt.figure(figsize=(14, 7))
plt.plot(df.index, df['Close'], 'b-', label='Precios Históricos')

for i, prices in enumerate(future_prices_list):
    if i == 0:
        plt.plot(future_dates, prices, 'r-', alpha=0.8, label='Precios Sintéticos')
    else:
        plt.plot(future_dates, prices, 'r-', alpha=0.2)

plt.title('Precios Históricos y Proyecciones Sintéticas', fontsize=14)
plt.xlabel('Fecha', fontsize=12)
plt.ylabel('Precio', fontsize=12)
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

## 5. Calcular el VaR mediante Simulación de Monte Carlo

In [ ]:
# Calcular el VaR a 1 día
initial_price = df['Close'].iloc[-1]
investment_amount = 100000  # Inversión de $100,000
position_size = investment_amount / initial_price  # Número de acciones

# Ajustar el VaR por el tamaño de la posición
var_mc, es_mc = monte_carlo_var(future_prices_list, initial_price, horizon=1, confidence_level=0.95)
var_mc_position = var_mc * position_size
es_mc_position = es_mc * position_size

print(f"VaR de Monte Carlo (95%) a 1 día: ${var_mc_position:.2f}")
print(f"Expected Shortfall (95%) a 1 día: ${es_mc_position:.2f}")

In [ ]:
# Comparar con otros métodos de cálculo de VaR

# Extraer rendimientos de un día para cada serie sintética
one_day_returns = [synth_df['Returns'].values[0] for synth_df in synthetic_df_list]

# VaR Paramétrico (distribución normal)
var_param_normal = parametric_var(one_day_returns, investment_amount, horizon=1, confidence_level=0.95, distribution='normal')

# VaR Paramétrico (distribución t-student)
var_param_t = parametric_var(one_day_returns, investment_amount, horizon=1, confidence_level=0.95, distribution='t-student')

# VaR Histórico
var_hist = historical_var(one_day_returns, investment_amount, confidence_level=0.95)

# Mostrar resultados
methods = ['Monte Carlo', 'Paramétrico (Normal)', 'Paramétrico (t-student)', 'Histórico']
var_values = [var_mc_position, var_param_normal, var_param_t, var_hist]

plot_multiple_var_methods(var_values, methods, title='Comparación de Métodos de VaR (95%) a 1 día')
plt.show()

## 6. Visualizar la Distribución de Pérdidas y Ganancias (P&L)

In [ ]:
# Calcular P&L para el horizonte de 1 día
one_day_prices = [prices[0] for prices in future_prices_list]
one_day_pnl = [(price - initial_price) * position_size for price in one_day_prices]

# Visualizar distribución de P&L y VaR
plot_var_histogram(one_day_pnl, var_mc_position, confidence_level=0.95, 
                   title='Distribución de P&L a 1 día y VaR (95%)')
plt.show()

## 7. Análisis de Sensibilidad del VaR

In [ ]:
# Calcular VaR para diferentes niveles de confianza
confidence_levels = [0.90, 0.95, 0.99]
var_results = []
es_results = []

for cl in confidence_levels:
    var, es = monte_carlo_var(future_prices_list, initial_price, horizon=1, confidence_level=cl)
    var_results.append(var * position_size)
    es_results.append(es * position_size)

# Visualizar resultados
plt.figure(figsize=(10, 6))
width = 0.35
ind = np.arange(len(confidence_levels))

plt.bar(ind - width/2, var_results, width, label='VaR', color='skyblue')
plt.bar(ind + width/2, es_results, width, label='Expected Shortfall', color='salmon')

plt.ylabel('USD', fontsize=12)
plt.title('VaR y Expected Shortfall por Nivel de Confianza', fontsize=14)
plt.xticks(ind, [f'{cl*100:.0f}%' for cl in confidence_levels])
plt.legend()
plt.grid(True, alpha=0.3, axis='y')

# Añadir valores encima de las barras
for i in range(len(confidence_levels)):
    plt.text(i - width/2, var_results[i] + 50, f'${var_results[i]:.2f}', ha='center')
    plt.text(i + width/2, es_results[i] + 50, f'${es_results[i]:.2f}', ha='center')

plt.tight_layout()
plt.show()

## 8. Conclusiones

En este notebook, hemos implementado un flujo completo para:

1. Preprocesar datos históricos de precios de acciones
2. Entrenar un modelo TimeGAN para generar series temporales sintéticas
3. Proyectar trayectorias futuras de precios
4. Calcular el VaR mediante simulación de Monte Carlo y compararlo con otros métodos

Los resultados muestran que el modelo TimeGAN es capaz de generar trayectorias sintéticas que capturan las características estadísticas de los datos históricos, lo que permite una estimación más robusta del riesgo.

El VaR a 1 día calculado mediante simulación de Monte Carlo proporciona una estimación conservadora del riesgo, en línea con los métodos paramétricos e históricos, pero con la ventaja de capturar mejor las propiedades temporales de la serie.